# Einsum Examples

`np.einsum` expresses tensor contractions in **Einstein summation notation**: you label every axis of every operand with a letter, then declare which labels survive into the output. Everything else follows from three rules.

| Rule | Meaning |
| --- | --- |
| A label repeated **across operands** | those axes are aligned and multiplied elementwise |
| A label **missing from the output** | that axis is summed (contracted) away |
| A label repeated **within one operand** | take the diagonal along those axes |

The order of the labels on the right of `->` sets the axis order of the result, so transposition is free — it is just a relabelling.

In full generality, for operands $A^{(1)}, \dots, A^{(P)}$ with output labels $S$ and contracted labels $T$:

$$
O_{S} = \sum_{T} \prod_{p=1}^{P} A^{(p)}_{\,\text{labels}(p)}
$$

Every operation below — matmul, transpose, trace, outer product, attention — is a special case of that one expression. The value of einsum is that it makes the contraction explicit instead of hiding it behind a function name and an axis convention.

> Each cell below is self-contained: it builds its own inputs, so cells can be run in any order. Small integer arrays are used throughout so the results can be checked by hand.

### Matrix Multiplication

$$
C_{ij} = \sum_{k} A_{ik} B_{kj}
$$

The label $k$ appears in both operands but not in the output, so it is the **contracted** axis: the two matrices are aligned along it and summed over. The labels $i$ and $j$ appear in the output, so they are carried through — one output element per $(i, j)$ pair.

Shapes must agree on the contracted label: $(M \times K) \cdot (K \times N) \rightarrow (M \times N)$. Cost is $\mathcal{O}(MKN)$ — one multiply-add per $(i, j, k)$ triple.

This is exactly what `A @ B` does; einsum just names the axes instead of relying on the positional convention that matmul contracts the last axis of the left operand against the second-to-last of the right.

In [ ]:
import numpy as np

A = np.arange(6).reshape(3, 2)      # (3, 2)
B = np.arange(8).reshape(2, 4)      # (2, 4)

M = np.einsum('ik,kj->ij', A, B)    # contract over k
print(M)
print(np.allclose(M, A @ B))        # same as the @ operator

### Outer Product

$$
M_{ij} = a_i b_j
$$

No label is shared between the operands, so **nothing is contracted** — every element of $a$ meets every element of $b$. The result gains an axis rather than losing one: $(M,) \times (N,) \rightarrow (M \times N)$.

This is the instructive contrast with the dot product further down. Same two vectors, same subscripts on the inputs; the only difference is whether the shared label survives into the output. Keep both labels and you get the outer product; share one label and drop it and you get the inner product.

In [ ]:
import numpy as np

a = np.array([1, 2, 3])             # (3,)
b = np.array([10, 20])              # (2,)

outer = np.einsum('i,j->ij', a, b)  # no shared label -> nothing summed
print(outer)
print(np.allclose(outer, np.outer(a, b)))

### Transpose

$$
(X^{\top})_{ji} = X_{ij}
$$

Pure relabelling: the same labels appear on both sides of the arrow, just in a different order. No multiplication, no summation — only a permutation of the axes. NumPy can satisfy this by adjusting strides rather than moving data.

The same trick generalizes to any permutation of any rank, which is where einsum beats chained `.transpose()` calls for readability: `'ijk->kij'` says what it does.

In [ ]:
import numpy as np

X = np.arange(6).reshape(2, 3)      # (2, 3)

Xt = np.einsum('ij->ji', X)         # reorder the labels
print(Xt)
print(np.allclose(Xt, X.T))

### Sum of All Elements

$$
s = \sum_{i} \sum_{j} X_{ij}
$$

The output subscript is **empty**, so every label is dropped and therefore summed. The result is a scalar — a full reduction over the array.

This is the limiting case of the "missing from the output means summed" rule: drop everything and nothing is left to index.

In [ ]:
import numpy as np

X = np.arange(6).reshape(2, 3)      # (2, 3)

total = np.einsum('ij->', X)        # empty output -> sum over i and j
print(total)
print(np.allclose(total, X.sum()))

### Column-wise Sum

$$
c_j = \sum_{i} X_{ij}
$$

A partial reduction: $i$ is dropped so it is summed over, while $j$ survives. Summing over the row index collapses the rows, leaving one value per column — shape $(M \times N) \rightarrow (N,)$.

Note that the notation names the axis you *keep*, whereas `X.sum(axis=0)` names the axis you *remove*. For rank-2 arrays that is a wash, but on a rank-5 tensor `'bhqkd->bhd'` is considerably easier to read than a tuple of axis numbers.

In [ ]:
import numpy as np

X = np.arange(6).reshape(2, 3)      # (2, 3)

col_sums = np.einsum('ij->j', X)    # i dropped -> summed over rows
print(col_sums)
print(np.allclose(col_sums, X.sum(axis=0)))

### Matrix Multiplication with a Transposed Operand

$$
C_{ik} = \sum_{j} X_{ij} V_{kj} \qquad \text{i.e.} \qquad C = X V^{\top}
$$

Here the shared label $j$ sits in the **second** axis of both operands, so the contraction runs along matching axes without either matrix needing to be transposed first. Shapes: $(M \times D)$ and $(N \times D) \rightarrow (M \times N)$.

This pattern is worth internalizing because it is the shape of a similarity matrix — every row of $X$ dotted against every row of $V$ — and therefore the core of the $QK^{\top}$ term in attention. Expressing it directly avoids materializing an intermediate transpose.

In [ ]:
import numpy as np

X = np.arange(6).reshape(2, 3)      # (2, 3) -- 2 rows of length 3
V = np.arange(9).reshape(3, 3)      # (3, 3) -- 3 rows of length 3

C = np.einsum('ij,kj->ik', X, V)    # contract over the shared trailing axis
print(C)
print(np.allclose(C, X @ V.T))

### Vector Dot Product

$$
s = \sum_{i} a_i b_i
$$

One shared label, dropped from the output: the vectors are multiplied elementwise and the result summed to a scalar.

Compare directly against the outer product above — `'i,j->ij'` versus `'i,i->'`. Reusing the *same* letter forces the axes to align pairwise instead of forming all combinations, and omitting it from the output collapses the result.

In [ ]:
import numpy as np

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

dot = np.einsum('i,i->', a, b)      # shared label, dropped -> scalar
print(dot)                          # 1*4 + 2*5 + 3*6 = 32
print(np.allclose(dot, np.dot(a, b)))

### Frobenius Inner Product

$$
\langle A, B \rangle_F = \sum_{i} \sum_{j} A_{ij} B_{ij} = \operatorname{tr}\!\left(A^{\top} B\right)
$$

The dot product generalized to matrices: multiply elementwise, then sum every entry. Both labels are shared and both are dropped, so a rank-2 pair reduces to a scalar.

Taking $B = A$ gives the squared Frobenius norm:

$$
\|A\|_F^2 = \sum_{i} \sum_{j} A_{ij}^2
$$

which is the quantity behind weight decay and most matrix-reconstruction losses.

In [ ]:
import numpy as np

A = np.arange(6).reshape(2, 3)
B = np.arange(6, 12).reshape(2, 3)

frob = np.einsum('ij,ij->', A, B)   # both labels shared and dropped
print(frob)
print(np.allclose(frob, np.trace(A.T @ B)))

sq_norm = np.einsum('ij,ij->', A, A)
print(sq_norm, np.allclose(sq_norm, np.linalg.norm(A) ** 2))

### Element-wise (Hadamard) Product

$$
(A \circ B)_{ij} = A_{ij} B_{ij}
$$

Both labels are shared — so the operands align elementwise — but both also appear in the output, so **nothing is summed**. Shape is preserved: $(M \times N) \rightarrow (M \times N)$.

Set this beside the Frobenius product above: identical input subscripts, and the only difference is whether the labels survive the arrow. Keeping them gives the elementwise product; dropping them sums that same product to a scalar. This is the clearest demonstration that in einsum, multiplication is decided by the inputs and summation is decided by the output.

In [ ]:
import numpy as np

A = np.arange(6).reshape(2, 3)
B = np.arange(6, 12).reshape(2, 3)

had = np.einsum('ij,ij->ij', A, B)  # labels kept -> no summation
print(had)
print(np.allclose(had, A * B))

### Diagonal Extraction

$$
d_i = A_{ii}
$$

The label $i$ is repeated **within a single operand**, which selects the entries where both axes carry the same index — the diagonal. This is the one rule that has no equivalent in ordinary matmul notation, and it works only on axes of equal length.

Drop the label from the output as well and the diagonal is summed, giving the trace:

$$
\operatorname{tr}(A) = \sum_{i} A_{ii}
$$

In [ ]:
import numpy as np

A = np.arange(9).reshape(3, 3)
print(A)

diag = np.einsum('ii->i', A)        # repeated label within one operand
print(diag)
print(np.allclose(diag, np.diag(A)))

trace = np.einsum('ii->', A)        # ... and dropped -> trace
print(trace, np.allclose(trace, np.trace(A)))

### Batched Matrix Multiplication

Given a stack of $B$ matrices $A_1, \dots, A_B$ with $A_b \in \mathbb{R}^{M \times K}$, and a second stack $V_1, \dots, V_B$ with $V_b \in \mathbb{R}^{K \times N}$, batched matmul applies an independent matrix product per batch element:

$$
C_b = A_b V_b \in \mathbb{R}^{M \times N}, \qquad b = 1, \dots, B
$$

The batch axis is a *spectator*: it is carried through untouched while the contraction happens only over the shared inner dimension. Total cost is $B$ times a single matmul, $\mathcal{O}(BMKN)$.

**Sources**

- [`numpy.matmul`](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html) — stacks of matrices are treated as elements residing in the last two indexes and broadcast accordingly.
- [`numpy.einsum`](https://numpy.org/doc/stable/reference/generated/numpy.einsum.html) — the general Einstein-summation interface.
- [cuBLAS `gemmStridedBatched`](https://docs.nvidia.com/cuda/cublas/#cublas-t-gemmstridedbatched) — the BLAS-level primitive this maps onto in practice.


### Attention

**Scaled dot-product attention.** With queries $Q \in \mathbb{R}^{n \times d_k}$, keys $K \in \mathbb{R}^{m \times d_k}$, and values $V \in \mathbb{R}^{m \times d_v}$:

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right) V
$$

The $1/\sqrt{d_k}$ factor counteracts the growth of the dot products with dimension, which would otherwise push the softmax into regions of vanishing gradient.

Written per row, with $\alpha_{ij}$ the attention weight of query $i$ over key $j$:

$$
\alpha_{ij} = \frac{\exp\!\left(q_i \cdot k_j / \sqrt{d_k}\right)}{\sum_{j'} \exp\!\left(q_i \cdot k_{j'} / \sqrt{d_k}\right)},
\qquad
o_i = \sum_j \alpha_{ij}\, v_j
$$

**Multi-head attention.** Each of the $h$ heads projects into its own subspace, attends independently, and the results are concatenated and mixed:

$$
\mathrm{head}_i = \mathrm{Attention}\!\left(QW_i^{Q},\, KW_i^{K},\, VW_i^{V}\right)
$$

$$
\mathrm{MultiHead}(Q, K, V) = \mathrm{Concat}(\mathrm{head}_1, \dots, \mathrm{head}_h)\, W^{O}
$$

with $W_i^{Q}, W_i^{K} \in \mathbb{R}^{d_{\text{model}} \times d_k}$, $W_i^{V} \in \mathbb{R}^{d_{\text{model}} \times d_v}$, and $W^{O} \in \mathbb{R}^{h d_v \times d_{\text{model}}}$. The original paper uses $h = 8$ and $d_k = d_v = d_{\text{model}}/h = 64$.

For causal (decoder) attention, an additive mask $M_{ij} = -\infty$ for $j > i$ is applied to the scores before the softmax.

**Sources**

- Vaswani et al., *Attention Is All You Need* (2017) — [arXiv:1706.03762](https://arxiv.org/abs/1706.03762). §3.2.1 scaled dot-product attention, §3.2.2 multi-head attention.
- Bahdanau, Cho & Bengio, *Neural Machine Translation by Jointly Learning to Align and Translate* (2014) — [arXiv:1409.0473](https://arxiv.org/abs/1409.0473). The additive-attention predecessor.
- Luong, Pham & Manning, *Effective Approaches to Attention-based Neural Machine Translation* (2015) — [arXiv:1508.04025](https://arxiv.org/abs/1508.04025). Introduces the multiplicative ("dot-product") form.


### Mixture of Experts

A MoE layer holds $n$ expert networks $E_1, \dots, E_n$ and a gating network $G$ that produces a distribution over them. The layer output is the gate-weighted combination of expert outputs:

$$
y = \sum_{i=1}^{n} G(x)_i \, E_i(x)
$$

Whenever $G(x)_i = 0$, expert $i$ need not be evaluated at all — this is where the conditional-computation savings come from.

**Noisy top-$k$ gating.** Softmax over a learned projection, with tunable Gaussian noise, keeping only the $k$ largest logits:

$$
H(x)_i = (x \cdot W_g)_i + \varepsilon \cdot \mathrm{softplus}\!\left((x \cdot W_{\text{noise}})_i\right), \qquad \varepsilon \sim \mathcal{N}(0, 1)
$$

$$
\mathrm{KeepTopK}(v, k)_i =
\begin{cases}
v_i & \text{if } v_i \text{ is in the top } k \text{ of } v \\
-\infty & \text{otherwise}
\end{cases}
$$

$$
G(x) = \mathrm{softmax}\!\left(\mathrm{KeepTopK}(H(x), k)\right)
$$

Sparsity makes $G(x)$ have at most $k$ nonzero entries, so cost per token is $\mathcal{O}(k)$ experts rather than $\mathcal{O}(n)$, while parameter count grows with $n$.

**Load balancing.** Left alone, gating collapses onto a few favoured experts. An auxiliary loss penalizes imbalance — in the sparsely-gated formulation, the squared coefficient of variation of the per-expert importance $\mathrm{Importance}(X) = \sum_{x \in X} G(x)$ over a batch:

$$
L_{\text{importance}}(X) = w_{\text{importance}} \cdot CV\!\left(\mathrm{Importance}(X)\right)^2
$$

Switch Transformer instead uses, with $f_i$ the fraction of tokens routed to expert $i$ and $P_i$ the mean gate probability for expert $i$:

$$
L_{\text{aux}} = \alpha \, n \sum_{i=1}^{n} f_i P_i
$$

**Sources**

- Jacobs, Jordan, Nowlan & Hinton, *Adaptive Mixtures of Local Experts* (1991), *Neural Computation* 3(1):79–87 — [doi:10.1162/neco.1991.3.1.79](https://doi.org/10.1162/neco.1991.3.1.79). The original MoE.
- Shazeer et al., *Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer* (2017) — [arXiv:1701.06538](https://arxiv.org/abs/1701.06538). §2.1 gating, noisy top-$k$; §4 load-balancing loss.
- Fedus, Zoph & Shazeer, *Switch Transformers* (2021) — [arXiv:2101.03961](https://arxiv.org/abs/2101.03961). Top-1 routing, capacity factor, the $\alpha n \sum f_i P_i$ auxiliary loss.
- Lepikhin et al., *GShard* (2020) — [arXiv:2006.16668](https://arxiv.org/abs/2006.16668). Top-2 routing and expert-parallel sharding.
